# Real GPU worker on KAGGLE — voice + face + lip-sync

**EXPERIMENTAL, untested on Kaggle specifically** (the code itself is the
same fixed gpu_worker.py/ingest.py/f5_finetune_driver.py already proven
against real Colab bugs this session — mediapipe removal, Piper's dead
piper-phonemize, an OpenCV dual-install conflict, and an OOM from F5-TTS's
own training script spawning 16 dataloader workers are all fixed at the code
level and won't recur here either. What's genuinely untested is Kaggle's own
base image — it may have its OWN new package-version surprises we haven't
hit yet; paste back any FAILED cell output same as always).

**Before running anything**, in the Kaggle notebook's right-hand **Settings**
panel:
1. **Internet: On** (off by default — needed for pip installs, git clone,
   downloading the F5-TTS/Whisper/SadTalker model weights, the cloudflared
   binary).
2. **Accelerator: GPU** (P100 or T4 x2 — whichever your quota offers;
   30 GPU-hours/week on the free tier, separate from Colab's own limit).

Keep the browser tab open and active — Kaggle idles out a notebook after a
period of inactivity, same risk as Colab. The app's **Training Studio** +
**Create Video** drive this worker once you paste the tunnel URL + token
into **Settings → Compute profiles → Kaggle → Washa Kaggle**.

| Profile / task | Real model | Notes |
|---|---|---|
| voice | faster-whisper (sw) + **F5-TTS fine-tune** | minutes–1h |
| face_identity | frame selection from your video + face detect | picks a real reference frame, no cartoon |
| face_performance | short driving clip from your video | for expression transfer |
| lipsync / generation | **SadTalker** (audio-driven) or LivePortrait | realism ∝ GPU + your video quality |

**Reality check:** on a free-tier GPU the talking head will look decent, not
indistinguishable. High realism needs an A100-class GPU. First results will
need tuning — paste FAILED errors back.


In [ ]:
#@title 1. GPU check
!nvidia-smi -L || print('NO GPU — Runtime > Change runtime type > T4/A100 GPU')


In [ ]:
#@title 1b. Kaggle prep: writable work dir
# Kaggle doesn't have /content by default (unlike Colab) — everything else
# in this notebook assumes it, so just create it once. /kaggle/working also
# exists and persists across the session if you'd rather use that instead.
!mkdir -p /content


In [ ]:
%%writefile /content/ingest.py
#!/usr/bin/env python3
"""
Training-data ingestion — brief §8.2. EXPERIMENTAL, GPU helps (Whisper).

Turns the founder's authorized videos into:
  dataset/wavs/*.wav          speech clips, 2.5–12 s, mono 22.05 kHz
  dataset/metadata.csv        LJSpeech-style  <id>|<text>
  dataset/metadata.jsonl      richer rows (duration, source, asr, quality)
  dataset/faces/*.jpg         sampled face crops (for the Phase 4 face profile)
  report.json                 per-video quality score + status

Model-agnostic: this output feeds whichever TTS we fine-tune AND the face
identity profile. Originals are never modified or deleted — only their paths
and sha1 are recorded (brief §8.2, §8.7).

Pipeline per video (brief §8.2):
  extract audio → split on silence → transcribe (Whisper) → align → quality
  score → (frames → detect face → face score) → add to dataset

Usage:
  python ingest.py --videos ./videos --out ./out [--whisper small] [--lang sw]
"""
from __future__ import annotations

import argparse
import csv
import hashlib
import json
import pathlib
import re
import subprocess
import sys

SR = 22050
MIN_SEC = 2.5
MAX_SEC = 12.0
VIDEO_EXT = {".mp4", ".mov", ".mkv", ".webm", ".avi", ".m4v", ".wav", ".m4a", ".mp3"}


def sh(cmd: list[str]) -> str:
    return subprocess.run(cmd, capture_output=True, text=True).stderr


def sha1(path: pathlib.Path) -> str:
    h = hashlib.sha1()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def extract_audio(src: pathlib.Path, dst: pathlib.Path) -> None:
    subprocess.run(
        ["ffmpeg", "-y", "-i", str(src), "-vn", "-ac", "1", "-ar", str(SR), str(dst)],
        check=True, capture_output=True,
    )


def duration(path: pathlib.Path) -> float:
    err = sh(["ffmpeg", "-i", str(path)])
    m = re.search(r"Duration:\s*(\d+):(\d+):(\d+\.\d+)", err)
    if not m:
        return 0.0
    return int(m[1]) * 3600 + int(m[2]) * 60 + float(m[3])


def adaptive_noise_db(mean_vol_db: float) -> float:
    """Silence threshold relative to the recording's own loudness, not a fixed
    number. A quiet phone-mic recording (mean around -30 to -35dB) has actual
    speech sitting close to or below a flat -30dB cutoff, so a fixed threshold
    misclassifies almost the whole file as "silence" and near-zero speech gets
    recognised. Set the cutoff clearly below the recording's average instead,
    clamped to a sane range."""
    return max(-45.0, min(-18.0, mean_vol_db - 10.0))


def silence_windows(wav: pathlib.Path, noise_db: float = -30, min_sil: float = 0.4):
    """Return (start, end) speech spans between detected silences.

    Natural speech is full of short pauses well under MIN_SEC on its own — a
    filter that drops any individual span shorter than MIN_SEC silently throws
    away most real conversational audio. Instead, short spans separated by a
    brief gap are MERGED into one clip (a pause inside a clip is normal); only
    a genuinely negligible leftover fragment (<0.6s) is dropped.
    """
    err = sh(["ffmpeg", "-i", str(wav), "-af",
              f"silencedetect=noise={noise_db}dB:d={min_sil}", "-f", "null", "-"])
    starts = [float(x) for x in re.findall(r"silence_start:\s*([\d.]+)", err)]
    ends = [float(x) for x in re.findall(r"silence_end:\s*([\d.]+)", err)]
    total = duration(wav)
    # speech = complement of the silence intervals
    sil = sorted(zip(starts, ends + [total] * (len(starts) - len(ends))))
    spans, cur = [], 0.0
    for s, e in sil:
        if s - cur > 0.3:
            spans.append((cur, s))
        cur = e
    if total - cur > 0.3:
        spans.append((cur, total))

    # merge spans forward across short gaps until each clip reaches MIN_SEC
    merged: list[tuple[float, float]] = []
    for s, e in spans:
        if merged and s - merged[-1][1] <= 1.2 and (merged[-1][1] - merged[-1][0]) < MIN_SEC:
            merged[-1] = (merged[-1][0], e)
        else:
            merged.append((s, e))

    # enforce length bounds: hard-cut > MAX, drop only negligible leftovers
    out = []
    for s, e in merged:
        while e - s > MAX_SEC:
            out.append((s, s + MAX_SEC))
            s += MAX_SEC
        if e - s >= 0.6:
            out.append((s, e))
    return out


def cut(wav: pathlib.Path, s: float, e: float, dst: pathlib.Path) -> None:
    subprocess.run(
        ["ffmpeg", "-y", "-i", str(wav), "-ss", f"{s:.3f}", "-to", f"{e:.3f}",
         "-ac", "1", "-ar", str(SR), str(dst)],
        check=True, capture_output=True,
    )


def mean_volume_db(wav: pathlib.Path) -> float:
    err = sh(["ffmpeg", "-i", str(wav), "-af", "volumedetect", "-f", "null", "-"])
    m = re.search(r"mean_volume:\s*(-?[\d.]+)\s*dB", err)
    return float(m[1]) if m else -99.0


# --------------------------------------------------------------------------- #

_whisper = None


def transcribe(wav: pathlib.Path, model_size: str, lang: str) -> tuple[str, float]:
    global _whisper
    if _whisper is None:
        from faster_whisper import WhisperModel  # type: ignore
        import torch  # noqa

        dev = "cuda" if _cuda() else "cpu"
        _whisper = WhisperModel(model_size, device=dev,
                                compute_type="float16" if dev == "cuda" else "int8")
    # vad_filter=False: we already pre-segment on silence with our own
    # loudness-adaptive detector (adaptive_noise_db) before a clip ever reaches
    # here. Whisper's *own* internal VAD (Silero, fixed sensitivity) applied on
    # top of that was a second, uncalibrated filter silently discarding real
    # speech in quiet recordings — it doesn't know this file's loudness.
    segs, _info = _whisper.transcribe(str(wav), language=lang, vad_filter=False)
    parts, logp = [], []
    for s in segs:
        parts.append(s.text.strip())
        logp.append(getattr(s, "avg_logprob", 0.0))
    text = re.sub(r"\s+", " ", " ".join(parts)).strip()
    conf = float(sum(logp) / len(logp)) if logp else -9.0
    return text, conf


def _cuda() -> bool:
    try:
        import torch

        return torch.cuda.is_available()
    except Exception:
        return False


# --------------------------------------------------------------------------- #

_facedet = None


def face_scores(video: pathlib.Path, out_dir: pathlib.Path, n: int = 12):
    """Sample n frames, return (found_ratio, mean_face_frac, mean_sharpness, saved).

    Uses OpenCV's Haar cascade (bundled with opencv-python, a stable decades-old
    API) rather than mediapipe's legacy `mp.solutions` detector, which recent
    mediapipe releases removed outright ('module mediapipe has no attribute
    solutions') — this sidesteps that version fragility entirely."""
    global _facedet
    try:
        import cv2  # type: ignore
    except Exception as exc:  # noqa: BLE001
        return {"error": f"face libs unavailable: {exc}"}

    if _facedet is None:
        _facedet = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

    cap = cv2.VideoCapture(str(video))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
    if total <= 0:
        return {"error": "no frames"}
    idxs = [int(total * (i + 0.5) / n) for i in range(n)]
    found, fracs, sharps, saved = 0, [], [], 0
    for k, fi in enumerate(idxs):
        cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
        ok, frame = cap.read()
        if not ok:
            continue
        h, w = frame.shape[:2]
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = _facedet.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))
        if len(faces) == 0:
            continue
        x, y, cw, ch = max(faces, key=lambda f: f[2] * f[3])
        found += 1
        fracs.append(max(0.0, min(1.0, ch / h)))
        crop = frame[y:y + ch, x:x + cw]
        if crop.size:
            sharps.append(cv2.Laplacian(cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY),
                                        cv2.CV_64F).var())
            if k < 6:
                cv2.imwrite(str(out_dir / f"{video.stem}_{k}.jpg"), crop)
                saved += 1
    cap.release()
    return {
        "found_ratio": round(found / max(1, len(idxs)), 2),
        "mean_face_frac": round(sum(fracs) / len(fracs), 3) if fracs else 0.0,
        "mean_sharpness": round(sum(sharps) / len(sharps), 1) if sharps else 0.0,
        "faces_saved": saved,
    }


# --------------------------------------------------------------------------- #

def classify(speech_sec: float, n_clips: int, vol_db: float, face: dict) -> tuple[int, str]:
    score = 10
    notes = []
    if vol_db < -34:
        score -= 3
        notes.append("TOO MUCH BACKGROUND NOISE / low level")
    if speech_sec < 60:
        score -= 3
        notes.append("NEEDS MORE SPEECH (<60s usable)")
    if n_clips < 8:
        score -= 1
    ff = face.get("found_ratio", 0.0)
    frac = face.get("mean_face_frac", 0.0)
    if "error" not in face:
        if ff < 0.5 or frac < 0.12:
            score -= 3
            notes.append("FACE NOT CLEAR ENOUGH")
        if face.get("mean_sharpness", 0) and face["mean_sharpness"] < 40:
            score -= 2
            notes.append("FACE BLURRY")
    score = max(1, score)
    status = "GOOD FOR TRAINING" if score >= 7 and not notes else (
        "; ".join(notes) if notes else "USABLE")
    return score, status


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--videos", required=True)
    ap.add_argument("--out", default="./out")
    ap.add_argument("--whisper", default="medium")  # "small" under-transcribes Swahili
    ap.add_argument("--lang", default="sw")
    ap.add_argument("--photos", default="", help="optional dir of face photos "
                    "(jpg/png) — used when you upload audio instead of video")
    a = ap.parse_args()

    vroot = pathlib.Path(a.videos)
    out = pathlib.Path(a.out)
    ds = out / "dataset"
    (ds / "wavs").mkdir(parents=True, exist_ok=True)
    (ds / "faces").mkdir(parents=True, exist_ok=True)
    (out / "work").mkdir(parents=True, exist_ok=True)

    photos_n = 0
    if a.photos and pathlib.Path(a.photos).is_dir():
        import shutil as _sh

        for p in sorted(pathlib.Path(a.photos).iterdir()):
            if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}:
                _sh.copy(p, ds / "faces" / p.name)
                photos_n += 1
        print(f"copied {photos_n} face photos -> dataset/faces/")

    vids = sorted(p for p in vroot.iterdir() if p.suffix.lower() in VIDEO_EXT)
    if not vids:
        sys.exit(f"no media in {vroot}")

    meta_csv = open(ds / "metadata.csv", "w", newline="", encoding="utf-8")
    meta_jsonl = open(ds / "metadata.jsonl", "w", encoding="utf-8")
    w = csv.writer(meta_csv, delimiter="|")
    report = []
    total_clips = 0

    for v in vids:
        print(f"\n=== {v.name} ===")
        full_wav = out / "work" / f"{v.stem}.wav"
        extract_audio(v, full_wav)
        vol_db = mean_volume_db(full_wav)
        spans = silence_windows(full_wav, noise_db=adaptive_noise_db(vol_db))
        print(f"  {len(spans)} candidate clips, mean volume {vol_db:.1f} dB")

        kept, speech_sec = 0, 0.0
        for i, (s, e) in enumerate(spans):
            clip_id = f"{v.stem}_{i:04d}"
            clip = ds / "wavs" / f"{clip_id}.wav"
            cut(full_wav, s, e, clip)
            text, conf = transcribe(clip, a.whisper, a.lang)
            if len(text) < 3 or conf < -2.2:
                clip.unlink(missing_ok=True)
                continue
            w.writerow([f"wavs/{clip_id}.wav", text])
            meta_jsonl.write(json.dumps({
                "id": clip_id, "wav": f"wavs/{clip_id}.wav", "text": text,
                "dur": round(e - s, 2), "asr_conf": round(conf, 3),
                "source": v.name,
            }, ensure_ascii=False) + "\n")
            kept += 1
            speech_sec += e - s
        total_clips += kept

        face = face_scores(v, ds / "faces") if v.suffix.lower() in {
            ".mp4", ".mov", ".mkv", ".webm", ".avi", ".m4v"} else {"error": "audio-only"}
        score, status = classify(speech_sec, kept, vol_db, face)
        print(f"  kept {kept} clips ({speech_sec:.0f}s speech) — score {score}/10 — {status}")
        report.append({
            "video": v.name, "sha1": sha1(v), "clips": kept,
            "speech_seconds": round(speech_sec, 1), "mean_volume_db": round(vol_db, 1),
            "face": face, "score": score, "status": status,
        })

    meta_csv.close()
    meta_jsonl.close()
    (out / "report.json").write_text(json.dumps({
        "clips_total": total_clips,
        "speech_seconds_total": round(sum(r["speech_seconds"] for r in report), 1),
        "face_photos_added": photos_n,
        "videos": report,
    }, indent=2, ensure_ascii=False))
    print(f"\nDONE — {total_clips} clips, "
          f"{sum(r['speech_seconds'] for r in report):.0f}s speech -> {ds}")
    print("Review dataset/metadata.csv (fix any bad transcripts) before fine-tuning.")
    for r in report:
        print(f"  {r['video']:40s} {r['score']}/10  {r['status']}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/gpu_worker.py
#!/usr/bin/env python3
"""
REAL GPU worker — voice only (Phase 3 / L1). EXPERIMENTAL, untested.

Implements worker/contract.md (generation + training) with real models:
  ingest        -> ffmpeg + faster-whisper (Swahili) -> clips + transcripts
  build_dataset -> assemble a dataset from ingested clips
  train         -> F5-TTS fine-tune (profile=voice only) -> ckpts/<modelRef>/model_last.pt
  evaluate      -> F5-TTS synth of a fixed script -> wav preview
  voice         -> F5-TTS synth with the PRODUCTION model (modelRef)
  face/lipsync  -> still MOCK here (real ones are Phase 4/5)

Voice was Piper until 2026-09; piper-phonemize has zero PyPI distributions
for Python 3.13 (unfixable via pinning), so voice training moved to F5-TTS
(MIT-ish, actively maintained, supports real fine-tuning not just zero-shot
conditioning). Needs the F5-TTS repo cloned + editable-installed so that its
own train scripts can resolve their package-relative data/ckpts dirs — see
F5TTS_REPO_DIR below and the install cell in gpu_worker.ipynb.

The app (Training Studio + Settings -> Compute) drives this over HTTP. Point
GPU_WORKER_URL at wherever this runs (Colab tunnel, Kaggle, a local NVIDIA box).

State lives under WORK_DIR (default /content/vs-work):
  videos/<sha1>/    per-recording clips + metadata      (from ingest)
  datasets/<ref>/   assembled dataset                    (from build_dataset)
  models/<ref>.f5.json  pointer to the F5-TTS checkpoint (from train)

Run:  python gpu_worker.py            (listens on :8800)
Env:  WORK_DIR, GPU_WORKER_TOKEN, PORT, FFMPEG, WHISPER_SIZE (default medium)
"""
from __future__ import annotations

import base64
import hashlib
import io
import json
import math
import os
import pathlib
import struct
import subprocess
import tarfile
import threading
import time
import uuid
import wave
import zlib
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

import ingest as ing  # sibling: extract_audio, silence_windows, cut, transcribe, mean_volume_db

WORK = pathlib.Path(os.environ.get("WORK_DIR", "/content/vs-work"))
TOKEN = os.environ.get("GPU_WORKER_TOKEN", "")
PORT = int(os.environ.get("PORT", "8800"))
WHISPER = os.environ.get("WHISPER_SIZE", "medium")  # "small" under-transcribes Swahili; T4 handles medium fine

JOBS: dict[str, dict] = {}
LOCK = threading.Lock()
for sub in ("videos", "datasets", "models", "tmp"):
    (WORK / sub).mkdir(parents=True, exist_ok=True)


# --------------------------------------------------------------------------- #
#  ingest                                                                    #
# --------------------------------------------------------------------------- #

def do_ingest(payload: dict, _set_stage=None):
    b64 = payload.get("fileB64")
    if not b64:
        raise ValueError("ingest needs fileB64")
    raw = base64.b64decode(b64)
    ref = hashlib.sha1(raw).hexdigest()[:16]
    vdir = WORK / "videos" / ref
    if (vdir / "metadata.csv").exists():
        meta = json.loads((vdir / "meta.json").read_text())
        return {"result": {"qualityScore": meta["qualityScore"],
                           "qualityStatus": meta["qualityStatus"], "meta": meta}}, None, None
    vdir.mkdir(parents=True, exist_ok=True)
    src = vdir / ("src" + pathlib.Path(payload.get("filename", "v.mp4")).suffix)
    src.write_bytes(raw)

    (vdir / "wavs").mkdir(exist_ok=True)
    full = vdir / "audio.wav"
    ing.extract_audio(src, full)
    vol = ing.mean_volume_db(full)
    spans = ing.silence_windows(full, noise_db=ing.adaptive_noise_db(vol))

    rows, speech = [], 0.0
    for i, (s, e) in enumerate(spans):
        cid = f"{ref}_{i:04d}"
        clip = vdir / "wavs" / f"{cid}.wav"
        ing.cut(full, s, e, clip)
        text, conf = ing.transcribe(clip, WHISPER, payload.get("lang", "sw"))
        if len(text) < 3 or conf < -2.2:
            clip.unlink(missing_ok=True)
            continue
        rows.append(f"{cid}|{text}")
        speech += e - s
    (vdir / "metadata.csv").write_text("\n".join(rows) + "\n", encoding="utf-8")

    # 60s assumed one long source video; the actual workflow is many SHORT
    # clips (brief §8.2 "variety helps" — several videos, not one marathon).
    # A ~60s clip with 20s+ of recognised speech is good content, not a
    # problem — only flag genuinely thin/near-silent recordings.
    score = 10
    status = "GOOD FOR TRAINING"
    if vol < -34:
        score -= 3; status = "TOO MUCH BACKGROUND NOISE"
    if speech < 15:
        score -= 3; status = "NEEDS MORE SPEECH"
    score = max(1, score)
    meta = {"workerRef": ref, "qualityScore": score, "qualityStatus": status,
            "clips": len(rows), "speech_seconds": round(speech),
            "est_speech_seconds": round(speech), "mean_volume_db": round(vol, 1)}
    (vdir / "meta.json").write_text(json.dumps(meta))
    return {"result": {"qualityScore": score, "qualityStatus": status, "meta": meta}}, None, None


# --------------------------------------------------------------------------- #
#  build_dataset                                                             #
# --------------------------------------------------------------------------- #

def do_build_dataset(payload: dict, _set_stage=None):
    refs = sorted(
        str((v.get("meta") or {}).get("workerRef", "")) for v in payload.get("videos", [])
    )
    refs = [r for r in refs if r]
    if not refs:
        raise ValueError("no ingested videos (missing workerRef) — re-run ingest against this worker")
    dref = hashlib.sha1("|".join(refs).encode()).hexdigest()[:16]
    ddir = WORK / "datasets" / dref
    (ddir / "wavs").mkdir(parents=True, exist_ok=True)

    rows, speech = [], 0.0
    for r in refs:
        vdir = WORK / "videos" / r
        if not (vdir / "metadata.csv").exists():
            continue
        for line in (vdir / "metadata.csv").read_text(encoding="utf-8").splitlines():
            if "|" not in line:
                continue
            cid, text = line.split("|", 1)
            wav = vdir / "wavs" / f"{cid}.wav"
            if not wav.exists():
                continue
            dst = ddir / "wavs" / f"{cid}.wav"
            if not dst.exists():
                dst.write_bytes(wav.read_bytes())
            rows.append(f"{cid}|{text}")
            speech += _wav_seconds(dst)
    (ddir / "metadata.csv").write_text("\n".join(rows) + "\n", encoding="utf-8")
    (ddir / "videos.json").write_text(json.dumps(refs))  # source refs for face/lipsync training
    return {"result": {"clipCount": len(rows), "speechSeconds": round(speech),
                       "frameCount": 0, "faceOkRatio": 0, "workerRef": dref}}, None, None


def _wav_seconds(p: pathlib.Path) -> float:
    try:
        with wave.open(str(p), "rb") as w:
            return w.getnframes() / float(w.getframerate())
    except Exception:
        return 0.0


# --------------------------------------------------------------------------- #
#  train  (F5-TTS fine-tune, voice only)                                     #
# --------------------------------------------------------------------------- #

FACE_MODEL = os.environ.get("FACE_MODEL", "sadtalker")  # sadtalker | liveportrait
SADTALKER_DIR = os.environ.get("SADTALKER_DIR", "/content/SadTalker")
LIVEPORTRAIT_DIR = os.environ.get("LIVEPORTRAIT_DIR", "/content/LivePortrait")

# F5-TTS's own train scripts resolve their data/ckpts dirs as
# "<installed f5_tts package dir>/../../{data,ckpts}" — that only lands inside
# the repo if F5-TTS was `pip install -e .`'d from a clone at this path (a
# plain `pip install f5-tts` from PyPI, fine for zero-shot inference, does
# NOT work for training). See gpu_worker.ipynb's install cell.
F5TTS_REPO_DIR = os.environ.get("F5TTS_REPO_DIR", "/content/F5-TTS")
F5_EXP_NAME = os.environ.get("F5_EXP_NAME", "F5TTS_v1_Base")
F5_TOKENIZER = os.environ.get("F5_TOKENIZER", "pinyin")  # matches the pretrained ckpt's vocab


def do_train(payload: dict, set_stage=None):
    profile = payload.get("profile")
    if profile in ("face_identity", "face_performance", "lipsync"):
        return _train_face(payload, set_stage)
    if profile == "speaking_style":
        raise ValueError("speaking_style training not implemented in gpu_worker (Phase 6)")
    if profile != "voice":
        raise ValueError(f"unknown profile {profile!r}")
    return _train_voice(payload, set_stage)


def _train_voice(payload: dict, set_stage=None):
    dref = str(payload.get("datasetRef") or "")
    ddir = WORK / "datasets" / dref
    if not (ddir / "metadata.csv").exists():
        raise ValueError("dataset not on this worker — rebuild the dataset against this worker")
    f5_pkg = pathlib.Path(F5TTS_REPO_DIR) / "src" / "f5_tts"
    if not f5_pkg.is_dir():
        raise ValueError(
            f"F5-TTS repo not found at {F5TTS_REPO_DIR} — clone it and `pip install -e .` "
            "first (see the install cell in gpu_worker.ipynb)"
        )

    model_ref = f"voice-{dref[:8]}-{int(time.time())}"
    tmp = WORK / "tmp" / model_ref
    tmp.mkdir(parents=True, exist_ok=True)

    # F5-TTS's data-prep script wants a CSV: "audio_file|text", absolute paths.
    rows = ["audio_file|text"]
    best_ref = None  # (duration, wav_path, text) — a short, clean clip for cloning conditioning
    for line in (ddir / "metadata.csv").read_text(encoding="utf-8").splitlines():
        if "|" not in line:
            continue
        cid, text = line.split("|", 1)
        text = text.strip()
        wav = (ddir / "wavs" / f"{cid}.wav").resolve()
        if not wav.exists() or not text:
            continue
        rows.append(f"{wav}|{text}")
        dur = _wav_seconds(wav)
        if 2.0 <= dur <= 12.0 and (best_ref is None or dur > best_ref[0]):
            best_ref = (dur, wav, text)
    if len(rows) < 2:
        raise ValueError("dataset has no usable clips — re-ingest with more speech")
    if best_ref is None:
        # no clip in the ideal 2-12s range — fall back to whatever exists
        cid, text = rows[1].split("|", 1)
    csv_path = tmp / "train.csv"
    csv_path.write_text("\n".join(rows) + "\n", encoding="utf-8")

    data_dir = pathlib.Path(F5TTS_REPO_DIR) / "data" / f"{model_ref}_{F5_TOKENIZER}"

    if set_stage:
        set_stage("preprocessing")
    subprocess.run(
        ["python", str(f5_pkg / "train/datasets/prepare_csv_wavs.py"), str(csv_path), str(data_dir)],
        check=True, capture_output=True, cwd=F5TTS_REPO_DIR,
    )

    if set_stage:
        set_stage("training")
    # Small founder-sized datasets (minutes, not hours) need far fewer updates
    # than F5-TTS's from-scratch defaults — save_per_updates/last_per_updates
    # default to 50000/5000, which a tiny dataset may NEVER reach, silently
    # producing no checkpoint at all. Keep both low so at least one save fires.
    #
    # Confirmed live (2026-09-12): with ~19 clips this dataset does ~10
    # updates/epoch. epochs=100 + save_every=50 meant 1000 updates and 20
    # checkpoint-save events (~1.2-2.4GB disk write each, slow on Colab) —
    # over an hour of wall time, mostly save I/O, not actual training compute.
    # Lower epochs + less frequent saves cuts that dramatically while still
    # giving several checkpoints across the run.
    epochs = int(os.environ.get("F5_EPOCHS", "40"))
    bs = int(os.environ.get("F5_BATCH_SIZE", "1400"))  # frames/gpu — conservative for a T4
    save_every = int(os.environ.get("F5_SAVE_EVERY", "100"))
    lr = os.environ.get("F5_LR", "1e-5")
    # 0 = no forked DataLoader worker processes. F5-TTS's stock script
    # hardcodes 16 (OOMs free Colab); even 2 still OOM'd live (confirmed via
    # dmesg) because forking AFTER the main process has loaded torch/CUDA/the
    # model makes each fork's copy-on-write pages balloon into private dirty
    # memory (~2.3GB/worker observed) — unrelated to dataset size.
    workers = int(os.environ.get("F5_NUM_WORKERS", "0"))
    driver = os.environ.get("F5_FINETUNE_DRIVER", "/content/f5_finetune_driver.py")
    if not pathlib.Path(driver).exists():
        raise ValueError(f"F5 finetune driver not found at {driver} — re-run the notebook's writefile cell")

    # Continue improving an existing voice instead of starting from F5-TTS's
    # base checkpoint every time — the app passes this when the founder picks
    # "improve" on an already-trained model (import_model puts its checkpoint
    # on this worker first if it isn't here already).
    resume_from = str(payload.get("resumeFromModelRef") or "")
    extra_args: list[str] = []
    if resume_from:
        resume_ptr = WORK / "models" / f"{resume_from}.f5.json"
        if not resume_ptr.exists():
            raise ValueError(
                f"resumeFromModelRef {resume_from!r} not on this worker — import it first"
            )
        resume_ckpt = json.loads(resume_ptr.read_text())["ckpt"]
        extra_args = ["--pretrain", resume_ckpt]

    # A long tqdm-heavy training run piped through capture_output=True buffers
    # its ENTIRE stdout/stderr in this process's memory until it exits — over
    # tens of minutes that alone can OOM the worker (killing the HTTP server
    # too, not just the training subprocess). Stream to a log file instead.
    log_path = tmp / "finetune.log"
    try:
        with open(log_path, "w") as logf:
            subprocess.run(
                ["accelerate", "launch", driver,
                 "--exp_name", F5_EXP_NAME, "--dataset_name", model_ref, "--finetune",
                 "--tokenizer", F5_TOKENIZER, "--epochs", str(epochs),
                 "--batch_size_per_gpu", str(bs), "--batch_size_type", "frame",
                 # F5-TTS defaults to -1 (keep every periodic checkpoint
                 # forever) — with save_per_updates this low, a long run
                 # fills the disk fast (confirmed live: one run alone left
                 # a 29GB ckpts/ dir on a free Colab box before this fix).
                 # 1 = keep only the newest periodic snapshot + model_last.pt.
                 "--keep_last_n_checkpoints", "1",
                 "--save_per_updates", str(save_every), "--last_per_updates", str(save_every),
                 "--learning_rate", lr, "--num_workers", str(workers), *extra_args],
                check=True, stdout=logf, stderr=subprocess.STDOUT, cwd=F5TTS_REPO_DIR,
            )
    except subprocess.CalledProcessError as exc:
        tail = log_path.read_text(errors="replace")[-3000:] if log_path.exists() else ""
        raise ValueError(f"F5-TTS finetune failed (exit {exc.returncode}):\n{tail}") from None

    if set_stage:
        set_stage("evaluating")
    ckpt_dir = pathlib.Path(F5TTS_REPO_DIR) / "ckpts" / model_ref
    ckpt = ckpt_dir / "model_last.pt"
    if not ckpt.exists():
        # last_per_updates may not have lined up exactly — fall back to the
        # newest periodic checkpoint rather than declaring total failure.
        numbered = [p for p in ckpt_dir.glob("model_*.pt")
                   if p.name != "model_last.pt" and not p.name.startswith("pretrained_")]
        numbered.sort(key=lambda p: int(p.stem.split("_")[1]) if p.stem.split("_")[1].isdigit() else -1)
        if not numbered:
            raise ValueError(
                f"training finished but produced no checkpoint under {ckpt_dir} — "
                "the dataset may be too small for even one save interval; "
                "lower F5_SAVE_EVERY and retrain"
            )
        ckpt = numbered[-1]
    vocab = data_dir / "vocab.txt"
    (WORK / "models" / f"{model_ref}.f5.json").write_text(json.dumps({
        "ckpt": str(ckpt), "vocab": str(vocab), "exp_name": F5_EXP_NAME,
        "ref_wav": str(best_ref[1]) if best_ref else "",
        "ref_text": best_ref[2] if best_ref else "",
    }))

    mins = float(payload.get("datasetStats", {}).get("speechSeconds", 0)) / 60
    proxy = max(1.0, min(9.0, round(8.4 * (1 - math.exp(-mins / 12)), 1)))
    return {"result": {"baseModel": f"f5-tts/{F5_EXP_NAME} (finetuned)",
                       "modelRef": model_ref, "evalScore": proxy,
                       "evalBreakdown": {"note_human_eval_required": proxy},
                       "gpuUsed": _gpu_name(), "license": "CC-BY-NC (F5-TTS weights) — check before commercial use",
                       "kind": "EXPERIMENTAL"}}, None, None


def _gpu_name() -> str:
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip()
        return out or "gpu"
    except Exception:
        return "gpu"


# --------------------------------------------------------------------------- #
#  face_identity / face_performance / lipsync training                        #
#  These do NOT fine-tune a network — they build a FACE PROFILE from your     #
#  real video (best reference frame + a driving clip) that the animation      #
#  model (SadTalker / LivePortrait) uses at generation time. Realism depends  #
#  on that model + your video quality + the GPU (brief §4 / §8.3).            #
# --------------------------------------------------------------------------- #

_face_cascade = None


def _get_face_cascade():
    """OpenCV Haar cascade — bundled with opencv-python, a stable decades-old
    API. mediapipe's legacy `mp.solutions` face detector was removed in recent
    mediapipe releases (breaking on the founder's worker: 'module mediapipe
    has no attribute solutions') — this avoids that version fragility
    entirely instead of chasing a pinned version."""
    global _face_cascade
    import cv2  # type: ignore

    if _face_cascade is None:
        path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
        _face_cascade = cv2.CascadeClassifier(path)
    return _face_cascade


def _detect_face(cv2mod, frame_bgr):
    """Returns the largest face as (x, y, w, h) in pixels, or None."""
    gray = cv2mod.cvtColor(frame_bgr, cv2mod.COLOR_BGR2GRAY)
    faces = _get_face_cascade().detectMultiScale(
        gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))
    if len(faces) == 0:
        return None
    # largest detected face = most likely the subject, not someone in the background
    return max(faces, key=lambda f: f[2] * f[3])


def _train_face(payload: dict, set_stage=None):
    import cv2  # type: ignore

    profile = payload["profile"]
    dref = str(payload.get("datasetRef") or "")
    ddir = WORK / "datasets" / dref
    vids_json = ddir / "videos.json"
    if not vids_json.exists():
        raise ValueError("dataset has no source videos on this worker — rebuild it here first")
    refs = json.loads(vids_json.read_text())

    model_ref = f"{profile}-{dref[:8]}-{int(time.time())}"
    fdir = WORK / "faces" / model_ref
    (fdir / "alts").mkdir(parents=True, exist_ok=True)

    if set_stage:
        set_stage("extracting_frames")
    best = []  # (score, frame_bgr, bbox)
    driving_src = None
    for r in refs:
        srcs = list((WORK / "videos" / r).glob("src.*"))
        if not srcs:
            continue
        driving_src = driving_src or srcs[0]
        cap = cv2.VideoCapture(str(srcs[0]))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
        for k in range(24):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(total * (k + 0.5) / 24))
            ok, frame = cap.read()
            if not ok:
                continue
            h, w = frame.shape[:2]
            face = _detect_face(cv2, frame)
            if face is None:
                continue
            x, y, fw, fh = [int(v) for v in face]
            if fw < 0.12 * w or fh < 0.12 * h:
                continue
            # front-ish (face centred) + sharp
            centred = 1 - min(1, abs((x + fw / 2) / w - 0.5) * 4)
            crop = frame[y:y + fh, x:x + fw]
            if crop.size == 0:
                continue
            sharp = cv2.Laplacian(cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var()
            best.append((centred * 2 + min(sharp, 400) / 100, frame, (x, y, fw, fh)))
        cap.release()

    if not best:
        raise ValueError("no clear, front-facing face found in the training video — "
                         "record closer / better lit, mark it, rebuild the dataset")
    best.sort(key=lambda t: t[0], reverse=True)

    if set_stage:
        set_stage("building_profile")
    # full-frame reference (SadTalker/LivePortrait want a head-and-shoulders image)
    cv2.imwrite(str(fdir / "reference.png"), best[0][1])
    for i, (_, fr, _) in enumerate(best[1:5]):
        cv2.imwrite(str(fdir / "alts" / f"{i}.png"), fr)

    # identity-consistency proxy: how alike the top faces are (embedding-free: hist corr)
    def _hist(fr, box):
        x, y, w0, h0 = box
        c = cv2.cvtColor(fr[y:y + h0, x:x + w0], cv2.COLOR_BGR2HSV)
        return cv2.calcHist([c], [0, 1], None, [30, 32], [0, 180, 0, 256])
    h0 = _hist(best[0][1], best[0][2])
    sims = [cv2.compareHist(h0, _hist(fr, bx), cv2.HISTCMP_CORREL) for _, fr, bx in best[1:6]]
    consistency = round(max(0.0, sum(sims) / max(1, len(sims))), 3)

    meta = {"profile": profile, "model_ref": model_ref, "face_model": FACE_MODEL,
            "reference": str(fdir / "reference.png"), "frames_scored": len(best),
            "identity_consistency": consistency}

    if profile in ("face_performance", "lipsync") and driving_src:
        if set_stage:
            set_stage("extracting_driving_clip")
        drv = fdir / "driving.mp4"
        ff = os.environ.get("FFMPEG", "ffmpeg")
        # a short natural talking segment for LivePortrait / performance transfer
        subprocess.run([ff, "-hide_banner", "-y", "-ss", "3", "-t", "6", "-i", str(driving_src),
                        "-an", "-vf", "scale=512:-2,fps=25", str(drv)], check=True, capture_output=True)
        meta["driving_clip"] = str(drv)

    (fdir / "meta.json").write_text(json.dumps(meta))

    proxy = round(2 + consistency * 6 + min(1.0, len(best) / 20), 1)
    return {"result": {"baseModel": FACE_MODEL, "modelRef": model_ref,
                       "evalScore": max(1.0, min(9.0, proxy)),
                       "evalBreakdown": {"identity_consistency": round(consistency * 10, 1),
                                         "frames_scored": min(10, len(best) / 2)},
                       "gpuUsed": _gpu_name(),
                       "license": "SadTalker Apache-2.0 / LivePortrait MIT",
                       "kind": "EXPERIMENTAL"}}, None, None


def _talking_head(reference_png: pathlib.Path, audio_wav: pathlib.Path,
                  driving_mp4: pathlib.Path | None = None) -> bytes:
    """Animate `reference_png` to `audio_wav`. SadTalker (audio-driven) by default."""
    out_dir = WORK / "tmp" / uuid.uuid4().hex
    out_dir.mkdir(parents=True, exist_ok=True)
    if FACE_MODEL == "liveportrait" and driving_mp4 and driving_mp4.exists():
        subprocess.run(["python", f"{LIVEPORTRAIT_DIR}/inference.py",
                        "-s", str(reference_png), "-d", str(driving_mp4),
                        "-o", str(out_dir)], check=True, capture_output=True, cwd=LIVEPORTRAIT_DIR)
        mp4s = sorted(out_dir.rglob("*.mp4"))
        vid = mp4s[-1]
        # mux the audio in (LivePortrait is video-driven, no audio)
        final = out_dir / "final.mp4"
        subprocess.run([os.environ.get("FFMPEG", "ffmpeg"), "-y", "-i", str(vid),
                        "-i", str(audio_wav), "-c:v", "copy", "-c:a", "aac",
                        "-shortest", str(final)], check=True, capture_output=True)
        return final.read_bytes()
    # SadTalker
    subprocess.run(["python", f"{SADTALKER_DIR}/inference.py",
                    "--source_image", str(reference_png),
                    "--driven_audio", str(audio_wav),
                    "--result_dir", str(out_dir),
                    "--still", "--preprocess", "full", "--enhancer", "gfpgan"],
                   check=True, capture_output=True, cwd=SADTALKER_DIR)
    mp4s = sorted(out_dir.rglob("*.mp4"))
    if not mp4s:
        raise ValueError("talking-head model produced no video")
    return mp4s[-1].read_bytes()


# --------------------------------------------------------------------------- #
#  evaluate / voice  (F5-TTS synth with the finetuned checkpoint)            #
# --------------------------------------------------------------------------- #

_f5_cache: dict[str, object] = {}


def _load_f5(ckpt_file: str, vocab_file: str, exp_name: str):
    if ckpt_file not in _f5_cache:
        from f5_tts.api import F5TTS  # local import: only needed once a voice model exists
        _f5_cache[ckpt_file] = F5TTS(model=exp_name, ckpt_file=ckpt_file, vocab_file=vocab_file)
    return _f5_cache[ckpt_file]


def _voice_pointer(model_ref: str) -> dict:
    p = WORK / "models" / f"{model_ref}.f5.json"
    if not p.exists():
        raise ValueError(f"voice model {model_ref} not found on this worker — train it here")
    return json.loads(p.read_text())


# --------------------------------------------------------------------------- #
#  export/import model — moves a trained voice between workers/sessions.     #
#  Every worker session (Colab/Kaggle/local) is ephemeral: its disk vanishes #
#  when that runtime disconnects, restarts, or you switch to a different     #
#  compute backend. Without this, every switch meant retraining from zero.   #
#  The app persists the exported bundle centrally and re-imports it onto     #
#  whichever worker is active before using/continuing the model there.      #
# --------------------------------------------------------------------------- #

def do_export_model(payload: dict, _set_stage=None):
    kind = str(payload.get("kind") or "")
    model_ref = str(payload.get("modelRef") or "")
    if kind != "voice":
        raise ValueError(f"export not implemented for kind {kind!r} yet")
    ptr = _voice_pointer(model_ref)
    ckpt = pathlib.Path(ptr["ckpt"])
    vocab = pathlib.Path(ptr["vocab"])
    if not ckpt.exists() or not vocab.exists():
        raise ValueError(f"voice model {model_ref}'s files are missing on this worker")
    manifest = {"exp_name": ptr.get("exp_name", F5_EXP_NAME), "ref_text": ptr.get("ref_text", "")}

    buf = io.BytesIO()
    with tarfile.open(fileobj=buf, mode="w:gz") as tar:
        tar.add(str(ckpt), arcname="model_last.pt")
        tar.add(str(vocab), arcname="vocab.txt")
        ref_wav = pathlib.Path(ptr["ref_wav"]) if ptr.get("ref_wav") else None
        if ref_wav and ref_wav.exists():
            tar.add(str(ref_wav), arcname="ref.wav")
        manifest_bytes = json.dumps(manifest).encode()
        info = tarfile.TarInfo(name="manifest.json")
        info.size = len(manifest_bytes)
        tar.addfile(info, io.BytesIO(manifest_bytes))
    data = buf.getvalue()
    return {"result": {"modelRef": model_ref, "kind": kind, "bytes": len(data)}}, data, "application/gzip"


def do_import_model(payload: dict, _set_stage=None):
    kind = str(payload.get("kind") or "")
    model_ref = str(payload.get("modelRef") or "")
    b64 = payload.get("bundleB64")
    if kind != "voice":
        raise ValueError(f"import not implemented for kind {kind!r} yet")
    if not model_ref or not b64:
        raise ValueError("import needs modelRef and bundleB64")

    ptr_file = WORK / "models" / f"{model_ref}.f5.json"
    if ptr_file.exists():
        return {"result": {"modelRef": model_ref, "already_present": True}}, None, None

    f5_pkg = pathlib.Path(F5TTS_REPO_DIR) / "src" / "f5_tts"
    if not f5_pkg.is_dir():
        raise ValueError(
            f"F5-TTS repo not found at {F5TTS_REPO_DIR} — clone it and `pip install -e .` "
            "first (see the install cell in gpu_worker.ipynb) before importing a model"
        )

    data = base64.b64decode(b64)
    ckpt_dir = pathlib.Path(F5TTS_REPO_DIR) / "ckpts" / model_ref
    data_dir = pathlib.Path(F5TTS_REPO_DIR) / "data" / f"{model_ref}_{F5_TOKENIZER}"
    ref_dir = WORK / "models" / f"{model_ref}.ref"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    data_dir.mkdir(parents=True, exist_ok=True)
    ref_dir.mkdir(parents=True, exist_ok=True)

    manifest: dict = {}
    ref_wav_path = ""
    # Extract by exact expected member name only — never a caller-supplied
    # path — so a crafted archive can't write outside these fixed locations.
    with tarfile.open(fileobj=io.BytesIO(data), mode="r:gz") as tar:
        names = set(tar.getnames())
        if "model_last.pt" in names:
            with tar.extractfile("model_last.pt") as f:
                (ckpt_dir / "model_last.pt").write_bytes(f.read())
        if "vocab.txt" in names:
            with tar.extractfile("vocab.txt") as f:
                (data_dir / "vocab.txt").write_bytes(f.read())
        if "ref.wav" in names:
            with tar.extractfile("ref.wav") as f:
                (ref_dir / "ref.wav").write_bytes(f.read())
            ref_wav_path = str(ref_dir / "ref.wav")
        if "manifest.json" in names:
            with tar.extractfile("manifest.json") as f:
                manifest = json.loads(f.read().decode())

    if not (ckpt_dir / "model_last.pt").exists() or not (data_dir / "vocab.txt").exists():
        raise ValueError("imported bundle is missing model_last.pt or vocab.txt")

    ptr_file.write_text(json.dumps({
        "ckpt": str(ckpt_dir / "model_last.pt"), "vocab": str(data_dir / "vocab.txt"),
        "exp_name": manifest.get("exp_name", F5_EXP_NAME),
        "ref_wav": ref_wav_path, "ref_text": manifest.get("ref_text", ""),
    }))
    return {"result": {"modelRef": model_ref, "imported": True}}, None, None


def _f5_synth(model_ref: str, text: str) -> bytes:
    ptr = _voice_pointer(model_ref)
    if not ptr.get("ref_wav"):
        raise ValueError(f"voice model {model_ref} has no reference clip recorded — retrain")
    tts = _load_f5(ptr["ckpt"], ptr["vocab"], ptr.get("exp_name", F5_EXP_NAME))
    out = WORK / "tmp" / f"{uuid.uuid4().hex}.wav"
    tts.infer(ref_file=ptr["ref_wav"], ref_text=ptr["ref_text"], gen_text=text, file_wave=str(out))
    data = out.read_bytes()
    out.unlink(missing_ok=True)
    return data


def _profile_dir(model_ref: str) -> pathlib.Path:
    d = WORK / "faces" / model_ref
    if not (d / "meta.json").exists():
        raise ValueError(f"face profile {model_ref} not found on this worker — train it here")
    return d


def _driving_clip_for(identity_dir: pathlib.Path, perf_ref: str) -> pathlib.Path | None:
    """A face_identity profile has no driving clip of its own (_train_face
    only extracts one for face_performance/lipsync). If the app trained
    face_performance/lipsync SEPARATELY from face_identity, prefer that
    profile's driving.mp4 — falling back to the identity profile's own
    (present when it WAS trained as face_performance/lipsync directly)."""
    if perf_ref:
        try:
            perf_drv = _profile_dir(perf_ref) / "driving.mp4"
            if perf_drv.exists():
                return perf_drv
        except ValueError:
            pass
    own_drv = identity_dir / "driving.mp4"
    return own_drv if own_drv.exists() else None


def _audio_for(script_text: str) -> pathlib.Path:
    """F5-TTS synth with the latest trained voice if there is one, else a mock tone."""
    models = sorted((WORK / "models").glob("voice-*.f5.json"))
    out = WORK / "tmp" / f"{uuid.uuid4().hex}.wav"
    if models:
        model_ref = models[-1].name[: -len(".f5.json")]
        out.write_bytes(_f5_synth(model_ref, script_text))
    else:
        out.write_bytes(_mock_wav(max(2.0, len(script_text.split()) / 2.3),
                                  max(1, len(script_text.split()))))
    return out


def do_evaluate(payload: dict, _set_stage=None):
    profile = payload.get("profile")
    if profile == "voice":
        data = _f5_synth(str(payload.get("modelRef") or ""), str(payload.get("scriptText", "")))
        return {"result": {"scores": {}, "note": "listen and score against §4",
                           "kind": "EXPERIMENTAL"}}, data, "audio/wav"

    if profile == "speaking_style":
        return {"result": {"scores": {}, "note": "not implemented"}}, _mock_wav(3, 6), "audio/wav"

    # face_identity / face_performance / lipsync -> visual preview
    d = _profile_dir(str(payload.get("modelRef") or ""))
    ref = d / "reference.png"
    if profile == "face_identity":
        # the point of this preview: "is this really me, not a cartoon?"
        return {"result": {"scores": {}, "note": "this is a real frame from your video — "
                           "check identity, skin, lighting (§4)", "kind": "EXPERIMENTAL"}}, \
               ref.read_bytes(), "image/png"
    audio = _audio_for(str(payload.get("scriptText", "")))
    drv = _driving_clip_for(d, str(payload.get("perfRef") or ""))
    video = _talking_head(ref, audio, drv)
    return {"result": {"scores": {}, "note": "watch: identity held? mouth matches Swahili? "
                       "believable as a real recording? (§4)", "kind": "EXPERIMENTAL"}}, \
           video, "video/mp4"


def do_voice(payload: dict, _set_stage=None):
    ref = str(payload.get("modelRef") or "")
    if not ref:
        raise ValueError("no PRODUCTION voice model — train one in the Training Studio and "
                         "promote it, or switch GPU provider back to local-mock")
    data = _f5_synth(ref, str(payload.get("text", "")))
    return data, "audio/wav", {"modelRef": ref}


# --------------------------------------------------------------------------- #
#  face / lipsync  — still MOCK here (Phase 4/5)                             #
# --------------------------------------------------------------------------- #

def _mock_wav(seconds: float, words: int) -> bytes:
    sr = 22050
    n = int(max(0.5, seconds) * sr)
    spw = max(1, n // max(1, words))
    pcm = bytearray()
    for i in range(n):
        wp = (i % spw) / spw
        env = 0.5 - 0.5 * math.cos((wp / 0.8) * 2 * math.pi) if wp < 0.8 else 0.0
        s = math.sin(2 * math.pi * 130 * i / sr) * env * 0.28
        pcm += struct.pack("<h", max(-32768, min(32767, int(s * 32767))))
    buf = WORK / "tmp" / f"{uuid.uuid4().hex}.wav"
    with wave.open(str(buf), "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(sr); w.writeframes(bytes(pcm))
    data = buf.read_bytes(); buf.unlink(missing_ok=True)
    return data


def do_face(payload: dict, _set_stage=None):
    ref = str(payload.get("modelRef") or "")
    if ref:
        d = _profile_dir(ref)
        return (d / "reference.png").read_bytes(), "image/png", {"modelRef": ref, "real": True}
    # no trained face profile yet -> obvious mock (brief §0)
    w = int(payload.get("width", 720)); h = int(payload.get("height", 900))
    return _mock_png(w, h), "image/png", {"width": w, "height": h, "mock": True}


def _mock_png(w: int, h: int) -> bytes:
    def chunk(t, d):
        return (struct.pack(">I", len(d)) + t + d +
                struct.pack(">I", zlib.crc32(t + d) & 0xFFFFFFFF))
    stride = w * 3
    raw = bytearray()
    for _ in range(h):
        raw.append(0)
        raw += bytes((150, 110, 84)) * w
    return (b"\x89PNG\r\n\x1a\n"
            + chunk(b"IHDR", struct.pack(">IIBBBBB", w, h, 8, 2, 0, 0, 0))
            + chunk(b"IDAT", zlib.compress(bytes(raw)))
            + chunk(b"IEND", b""))


def do_lipsync(payload: dict, _set_stage=None):
    audio_b64 = payload.get("audioB64")
    if not audio_b64:
        raise ValueError("lipsync needs audioB64")
    ap = WORK / "tmp" / f"{uuid.uuid4().hex}.wav"
    ap.write_bytes(base64.b64decode(audio_b64))

    ref_model = str(payload.get("modelRef") or "")
    if ref_model:
        # real talking-head: your trained face + this audio (SadTalker / LivePortrait)
        d = _profile_dir(ref_model)
        drv = _driving_clip_for(d, str(payload.get("perfRef") or ""))
        video = _talking_head(d / "reference.png", ap, drv)
        ap.unlink(missing_ok=True)
        return video, "video/mp4", {"modelRef": ref_model, "real": True, "faceModel": FACE_MODEL}

    # no trained face profile -> obvious mock lip-bar over the sent face
    face_b64 = payload.get("faceB64")
    if not face_b64:
        raise ValueError("no face profile and no faceB64 — train a face profile first")
    w = int(payload.get("width", 720)); h = int(payload.get("height", 900))
    fp = WORK / "tmp" / f"{uuid.uuid4().hex}.png"; fp.write_bytes(base64.b64decode(face_b64))
    op = WORK / "tmp" / f"{uuid.uuid4().hex}.mp4"
    ff = os.environ.get("FFMPEG", "ffmpeg")
    vf = (f"scale={w}:{h},drawbox=x={round(w/2-w*0.12)}:y={round(h*0.72)}:"
          f"w={round(w*0.24)}:h='{round(h*0.09)}*abs(sin(2*PI*t*3))':color=black@0.85:t=fill,"
          f"format=yuv420p")
    subprocess.run([ff, "-hide_banner", "-y", "-loop", "1", "-i", str(fp), "-i", str(ap),
                    "-shortest", "-vf", vf, "-r", "25", "-c:v", "libx264", "-preset", "ultrafast",
                    "-tune", "stillimage", "-c:a", "aac", "-pix_fmt", "yuv420p", str(op)],
                   check=True, capture_output=True)
    data = op.read_bytes()
    for p in (fp, ap, op):
        p.unlink(missing_ok=True)
    return data, "video/mp4", {"width": w, "height": h, "mock_lipbar": True}


# --------------------------------------------------------------------------- #
#  dispatch + http                                                           #
# --------------------------------------------------------------------------- #

BINARY = {"voice": do_voice, "face": do_face, "lipsync": do_lipsync}
JSON_TASKS = {"ingest": do_ingest, "build_dataset": do_build_dataset,
              "train": do_train, "evaluate": do_evaluate,
              "export_model": do_export_model, "import_model": do_import_model}


def _process(job_id: str, task: dict) -> None:
    with LOCK:
        JOBS[job_id]["status"] = "running"
    ttype = task.get("type")
    payload = task.get("payload", {})

    def set_stage(s):
        with LOCK:
            JOBS[job_id]["stage"] = s

    try:
        if ttype in BINARY:
            data, mime, meta = BINARY[ttype](payload)
            with LOCK:
                JOBS[job_id].update(status="done", artifact=data, mime=mime,
                                    kind="EXPERIMENTAL", model=f"gpu-{ttype}", meta=meta)
            return
        body, data, mime = JSON_TASKS[ttype](payload, set_stage)
        with LOCK:
            JOBS[job_id].update(status="done", result=body.get("result", {}),
                                artifact=data, mime=mime, kind="EXPERIMENTAL",
                                model=f"gpu-{ttype}")
    except subprocess.CalledProcessError as exc:  # noqa: BLE001
        tail = (exc.stderr or b"")[-1500:].decode(errors="replace") if isinstance(exc.stderr, bytes) else str(exc.stderr)
        with LOCK:
            JOBS[job_id].update(status="error", error=f"{exc}\n{tail}")
    except Exception as exc:  # noqa: BLE001
        with LOCK:
            JOBS[job_id].update(status="error", error=str(exc))


class H(BaseHTTPRequestHandler):
    def _auth(self):
        return not TOKEN or self.headers.get("Authorization") == f"Bearer {TOKEN}"

    def _j(self, code, body):
        b = json.dumps(body).encode()
        self.send_response(code); self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(b))); self.end_headers(); self.wfile.write(b)

    def log_message(self, *_):
        pass

    def do_GET(self):  # noqa: N802
        if self.path == "/health":
            return self._j(200, {"ok": True, "impl": "gpu_worker", "gpu": _gpu_name()})
        if not self._auth():
            return self._j(401, {"error": "unauthorized"})
        if self.path.startswith("/jobs/"):
            with LOCK:
                job = JOBS.get(self.path.split("/", 2)[2])
            if not job:
                return self._j(404, {"error": "no such job"})
            if job["status"] == "done":
                r = {"status": "done", "kind": job.get("kind"), "model": job.get("model"),
                     "meta": job.get("meta", {})}
                if job.get("artifact") is not None:
                    r["artifactUrl"] = f"/artifacts/{self.path.split('/',2)[2]}"
                    r["mime"] = job.get("mime")
                if "result" in job:
                    r["result"] = job["result"]
                return self._j(200, r)
            if job["status"] == "error":
                return self._j(200, {"status": "error", "error": job.get("error", "")})
            return self._j(200, {"status": job["status"], "stage": job.get("stage", "")})
        if self.path.startswith("/artifacts/"):
            with LOCK:
                job = JOBS.get(self.path.split("/", 2)[2])
            if not job or job.get("status") != "done" or job.get("artifact") is None:
                return self._j(404, {"error": "not ready"})
            self.send_response(200); self.send_header("Content-Type", job["mime"])
            self.send_header("Content-Length", str(len(job["artifact"]))); self.end_headers()
            self.wfile.write(job["artifact"]); return
        return self._j(404, {"error": "not found"})

    def do_POST(self):  # noqa: N802
        if not self._auth():
            return self._j(401, {"error": "unauthorized"})
        if self.path != "/run":
            return self._j(404, {"error": "not found"})
        n = int(self.headers.get("Content-Length", "0"))
        task = json.loads(self.rfile.read(n) or b"{}")
        if task.get("type") not in BINARY and task.get("type") not in JSON_TASKS:
            return self._j(400, {"error": f"unknown task {task.get('type')!r}"})
        jid = uuid.uuid4().hex
        with LOCK:
            JOBS[jid] = {"status": "queued"}
        threading.Thread(target=_process, args=(jid, task), daemon=True).start()
        return self._j(202, {"jobId": jid})


if __name__ == "__main__":
    print(f"[gpu_worker] :{PORT}  WORK={WORK}  gpu={_gpu_name()}  auth={'on' if TOKEN else 'off'}")
    ThreadingHTTPServer(("0.0.0.0", PORT), H).serve_forever()


In [ ]:
%%writefile /content/f5_finetune_driver.py
"""
Colab-safe replacement for F5-TTS's own finetune_cli.py.

Identical to the stock script EXCEPT: adds --num_workers (default 0) and
threads it through to Trainer.train(). The stock script hardcodes
num_workers=16 as Trainer.train()'s Python default (no CLI flag exists for
it). Two confirmed-live OOM kills so far, both via dmesg's
"Memory cgroup out of memory" / oom-kill on this same worker process:
  1. num_workers=16 (the stock default) — too many DataLoader workers for
     free Colab, ~38min to OOM.
  2. num_workers=2 (first fix) — STILL OOM'd, ~2min in, right after saving
     a checkpoint at update 100. Root cause: DataLoader workers are forked
     AFTER the main process has already loaded torch + CUDA context + the
     model, so each forked worker's copy-on-write pages get touched by
     Python's refcounting and become private dirty memory — ~2.3GB RSS per
     worker observed in dmesg, not proportional to the (tiny) dataset.
num_workers=0 avoids forking DataLoader workers entirely (main-process-only
loading), which removes this class of duplication outright — the dataset
here is small enough that it costs little throughput.
"""
import argparse
import os
import shutil
from importlib.resources import files

from cached_path import cached_path

from f5_tts.model import CFM, DiT, Trainer, UNetT
from f5_tts.model.dataset import load_dataset
from f5_tts.model.utils import get_tokenizer

# Trainer.train() hardcodes persistent_workers=True on its DataLoader, which
# PyTorch refuses to combine with num_workers=0 ("persistent_workers option
# needs num_workers > 0") — confirmed live: exactly this ValueError killed a
# 0-worker attempt outright. Patch the DataLoader trainer.py actually calls
# (its own module-level import, not torch.utils.data's — patching that would
# be too late, trainer.py already bound the name at its own import time) so
# num_workers=0 forces persistent_workers off instead of erroring.
import f5_tts.model.trainer as _f5trainer  # noqa: E402

_StockDataLoader = _f5trainer.DataLoader


class _SafeDataLoader(_StockDataLoader):
    def __init__(self, *args, **kwargs):
        if kwargs.get("num_workers", 0) == 0:
            kwargs["persistent_workers"] = False
            kwargs.pop("prefetch_factor", None)  # also invalid with num_workers=0
        super().__init__(*args, **kwargs)


_f5trainer.DataLoader = _SafeDataLoader


# -------------------------- Dataset Settings --------------------------- #
target_sample_rate = 24000
n_mel_channels = 100
hop_length = 256
win_length = 1024
n_fft = 1024
mel_spec_type = "vocos"  # 'vocos' or 'bigvgan'


# -------------------------- Argument Parsing --------------------------- #
def parse_args():
    parser = argparse.ArgumentParser(description="Train CFM Model")

    parser.add_argument(
        "--exp_name",
        type=str,
        default="F5TTS_v1_Base",
        choices=["F5TTS_v1_Base", "F5TTS_Base", "E2TTS_Base"],
        help="Experiment name",
    )
    parser.add_argument("--dataset_name", type=str, default="Emilia_ZH_EN", help="Name of the dataset to use")
    parser.add_argument("--learning_rate", type=float, default=1e-5, help="Learning rate for training")
    parser.add_argument("--batch_size_per_gpu", type=int, default=3200, help="Batch size per GPU")
    parser.add_argument(
        "--batch_size_type", type=str, default="frame", choices=["frame", "sample"], help="Batch size type"
    )
    parser.add_argument("--max_samples", type=int, default=64, help="Max sequences per batch")
    parser.add_argument("--grad_accumulation_steps", type=int, default=1, help="Gradient accumulation steps")
    parser.add_argument("--max_grad_norm", type=float, default=1.0, help="Max gradient norm for clipping")
    parser.add_argument("--epochs", type=int, default=100, help="Number of training epochs")
    parser.add_argument("--num_warmup_updates", type=int, default=20000, help="Warmup updates")
    parser.add_argument("--save_per_updates", type=int, default=50000, help="Save checkpoint every N updates")
    parser.add_argument(
        "--keep_last_n_checkpoints",
        type=int,
        default=-1,
        help="-1 to keep all, 0 to not save intermediate, > 0 to keep last N checkpoints",
    )
    parser.add_argument("--last_per_updates", type=int, default=5000, help="Save last checkpoint every N updates")
    parser.add_argument("--finetune", action="store_true", help="Use Finetune")
    parser.add_argument("--pretrain", type=str, default=None, help="the path to the checkpoint")
    parser.add_argument(
        "--tokenizer", type=str, default="pinyin", choices=["pinyin", "char", "custom"], help="Tokenizer type"
    )
    parser.add_argument(
        "--tokenizer_path",
        type=str,
        default=None,
        help="Path to custom tokenizer vocab file (only used if tokenizer = 'custom')",
    )
    parser.add_argument(
        "--log_samples",
        action="store_true",
        help="Log inferenced samples per ckpt save updates",
    )
    parser.add_argument("--logger", type=str, default=None, choices=[None, "wandb", "tensorboard"], help="logger")
    parser.add_argument(
        "--bnb_optimizer",
        action="store_true",
        help="Use 8-bit Adam optimizer from bitsandbytes",
    )
    parser.add_argument(
        "--num_workers", type=int, default=0,
        help="DataLoader worker processes (stock script hardcodes 16 — too many for free Colab)",
    )

    return parser.parse_args()


# -------------------------- Training Settings -------------------------- #


def main():
    args = parse_args()

    checkpoint_path = str(files("f5_tts").joinpath(f"../../ckpts/{args.dataset_name}"))

    # Model parameters based on experiment name

    if args.exp_name == "F5TTS_v1_Base":
        wandb_resume_id = None
        model_cls = DiT
        model_cfg = dict(
            dim=1024,
            depth=22,
            heads=16,
            ff_mult=2,
            text_dim=512,
            conv_layers=4,
        )
        if args.finetune:
            if args.pretrain is None:
                ckpt_path = str(cached_path("hf://SWivid/F5-TTS/F5TTS_v1_Base/model_1250000.safetensors"))
            else:
                ckpt_path = args.pretrain

    elif args.exp_name == "F5TTS_Base":
        wandb_resume_id = None
        model_cls = DiT
        model_cfg = dict(
            dim=1024,
            depth=22,
            heads=16,
            ff_mult=2,
            text_dim=512,
            text_mask_padding=False,
            conv_layers=4,
            pe_attn_head=1,
        )
        if args.finetune:
            if args.pretrain is None:
                ckpt_path = str(cached_path("hf://SWivid/F5-TTS/F5TTS_Base/model_1200000.pt"))
            else:
                ckpt_path = args.pretrain

    elif args.exp_name == "E2TTS_Base":
        wandb_resume_id = None
        model_cls = UNetT
        model_cfg = dict(
            dim=1024,
            depth=24,
            heads=16,
            ff_mult=4,
            text_mask_padding=False,
            pe_attn_head=1,
        )
        if args.finetune:
            if args.pretrain is None:
                ckpt_path = str(cached_path("hf://SWivid/E2-TTS/E2TTS_Base/model_1200000.pt"))
            else:
                ckpt_path = args.pretrain

    if args.finetune:
        if not os.path.isdir(checkpoint_path):
            os.makedirs(checkpoint_path, exist_ok=True)

        file_checkpoint = os.path.basename(ckpt_path)
        if not file_checkpoint.startswith("pretrained_"):  # Change: Add 'pretrained_' prefix to copied model
            file_checkpoint = "pretrained_" + file_checkpoint
        file_checkpoint = os.path.join(checkpoint_path, file_checkpoint)
        if not os.path.isfile(file_checkpoint):
            shutil.copy2(ckpt_path, file_checkpoint)
            print("copy checkpoint for finetune")

    # Use the tokenizer and tokenizer_path provided in the command line arguments

    tokenizer = args.tokenizer
    if tokenizer == "custom":
        if not args.tokenizer_path:
            raise ValueError("Custom tokenizer selected, but no tokenizer_path provided.")
        tokenizer_path = args.tokenizer_path
    else:
        tokenizer_path = args.dataset_name

    vocab_char_map, vocab_size = get_tokenizer(tokenizer_path, tokenizer)

    print("\nvocab : ", vocab_size)
    print("\nvocoder : ", mel_spec_type)

    mel_spec_kwargs = dict(
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=win_length,
        n_mel_channels=n_mel_channels,
        target_sample_rate=target_sample_rate,
        mel_spec_type=mel_spec_type,
    )

    model = CFM(
        transformer=model_cls(**model_cfg, text_num_embeds=vocab_size, mel_dim=n_mel_channels),
        mel_spec_kwargs=mel_spec_kwargs,
        vocab_char_map=vocab_char_map,
    )

    trainer = Trainer(
        model,
        args.epochs,
        args.learning_rate,
        num_warmup_updates=args.num_warmup_updates,
        save_per_updates=args.save_per_updates,
        keep_last_n_checkpoints=args.keep_last_n_checkpoints,
        checkpoint_path=checkpoint_path,
        batch_size_per_gpu=args.batch_size_per_gpu,
        batch_size_type=args.batch_size_type,
        max_samples=args.max_samples,
        grad_accumulation_steps=args.grad_accumulation_steps,
        max_grad_norm=args.max_grad_norm,
        logger=args.logger,
        wandb_project=args.dataset_name,
        wandb_run_name=args.exp_name,
        wandb_resume_id=wandb_resume_id,
        log_samples=args.log_samples,
        last_per_updates=args.last_per_updates,
        bnb_optimizer=args.bnb_optimizer,
    )

    train_dataset = load_dataset(args.dataset_name, tokenizer, mel_spec_kwargs=mel_spec_kwargs)

    trainer.train(
        train_dataset,
        num_workers=args.num_workers,
        resumable_with_seed=666,  # seed for shuffling dataset
    )


if __name__ == "__main__":
    main()


In [ ]:
#@title 3a. Install: voice (whisper + F5-TTS finetune)
!apt-get -qq install espeak-ng > /dev/null
!pip -q install faster-whisper opencv-python-headless soundfile
!git clone -q https://github.com/SWivid/F5-TTS /content/F5-TTS || true
%cd /content/F5-TTS
!pip -q install -e .
%cd /content
print('voice stack installed')


In [ ]:
#@title 3b. Install: face + lip-sync (SadTalker) — heavy, ~10-15 min
INSTALL_FACE = True  #@param {type:"boolean"}
if INSTALL_FACE:
    !git clone -q https://github.com/OpenTalker/SadTalker /content/SadTalker || true
    %cd /content/SadTalker
    !pip -q install -r requirements.txt
    !bash scripts/download_models.sh
    %cd /content
    # optional higher-quality video-driven model
    !git clone -q https://github.com/KwaiVGI/LivePortrait /content/LivePortrait || true
    print('face stack installed. Set FACE_MODEL=liveportrait in cell 5 to use LivePortrait.')


In [ ]:
#@title 4. (optional) worker token
import os, secrets
USE_TOKEN = True  #@param {type:"boolean"}
if USE_TOKEN:
    os.environ['GPU_WORKER_TOKEN'] = secrets.token_urlsafe(24)
    print('WORKER TOKEN (paste into Settings → Compute):', os.environ['GPU_WORKER_TOKEN'])


In [ ]:
#@title 5. Start the worker + public URL
import subprocess, threading, time, os
os.environ.setdefault('WORK_DIR', '/content/vs-work')
os.environ.setdefault('F5TTS_REPO_DIR', '/content/F5-TTS')
os.environ.setdefault('F5_EPOCHS', '100')     # lower for a quick first test
os.environ.setdefault('FACE_MODEL', 'sadtalker')  # or 'liveportrait'
# SadTalker's deps (basicsr/facexlib/gfpgan) pull in opencv-python (full),
# clobbering the opencv-python-headless we installed in 3a — having both
# installed together is a known OpenCV packaging conflict that breaks cv2
# (symptom: "module 'cv2' has no attribute 'CascadeClassifier'"). Force a
# single clean headless install right before starting the worker.
!pip -q uninstall -y opencv-python opencv-contrib-python opencv-python-headless opencv-contrib-python-headless > /dev/null 2>&1
!pip -q install --no-cache-dir opencv-python-headless > /dev/null
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
threading.Thread(target=lambda: subprocess.run(
    ['python', '/content/gpu_worker.py'], env={**os.environ, 'PORT': '8800'}), daemon=True).start()
time.sleep(4)
!cloudflared tunnel --url http://localhost:8800


## Use it

1. Copy the `https://xxx.trycloudflare.com` URL + token → **Settings → Compute
   profiles → Kaggle** → Hifadhi → **Washa Kaggle** (switches both providers
   to `http`/`worker` for you — no need to touch the raw fields below it).
2. **Training → Videos**: upload your videos, mark **Add to dataset**.
3. **Training → Datasets**: Build dataset.
4. **Training → Jobs**: run **Train** for `voice`, then `face_identity`, then
   `face_performance`, then `lipsync` — all from the **same** videos.
5. **Training → Models**: for each version, open it, watch/listen to the
   evaluation clips, and **Promote to PRODUCTION** the ones that pass §4
   ("would a Tanzanian viewer believe this is a real recording?").
6. **Create Video**: your script → your voice + your face + real lip-sync.

If face_identity's reference frame looks like a cartoon or isn't clearly you,
your training video is too low-res / dark / far — record better and rebuild.

**Switching back to Colab later**: once its GPU limit resets, just go to
**Settings → Compute profiles → Colab → Washa Colab** — no re-pasting either,
as long as that tunnel is still alive (if not, run a fresh Colab session and
save its new URL/token into the Colab profile first).
